# 🚨 ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ
## Emergency Vehicle Detection System — Colab Pro

**ໂຄງສ້າງ UI ໃໝ່:**
- About Bar (ຄົງທີ່ ด้านบน) + Sidebar ≥ 220 px / ≤ 60 px (collapse ໄດ້)
- White theme · ພາສາລາວ · YOLOv8x / YOLOv8m
- mAP50 = **97.54 %** · Precision = **95.03 %** · Recall = **96.00 %**

▶ **Run All** ➜ ເຮັດ setup ທັງໝົດ ➜ ລັນ FastAPI ➜ ໄດ້ ngrok URL


## 1️⃣  ກວດສອບ GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


## 2️⃣  ຕິດຕັ້ງ Dependencies

In [ ]:
!pip install -q fastapi uvicorn[standard] ultralytics opencv-python-headless pyngrok python-multipart jinja2 aiofiles
print('✅ ຕິດຕັ້ງສຳເລັດ')


## 3️⃣  Mount Google Drive + ກວດສອບໂມເດລ

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BASE = '/content/drive/MyDrive/FYP_25-26/runs'
MODEL_X = os.path.join(DRIVE_BASE, 'best_x.pt')
MODEL_M = os.path.join(DRIVE_BASE, 'best_m.pt')

print(f'YOLOv8x  → {MODEL_X}  [{"✅ ພົບ" if os.path.exists(MODEL_X) else "⚠️ ບໍ່ພົບ (ໃຊ້ pretrained)"}]')
print(f'YOLOv8m  → {MODEL_M}  [{"✅ ພົບ" if os.path.exists(MODEL_M) else "⚠️ ບໍ່ພົບ (ໃຊ້ pretrained)"}]')


## 4️⃣  ສ້າງໄຟລ Backend (config · detector · main)

In [ ]:
%%writefile config.py
import os

# ===== ໂມເດລ =====
DRIVE_BASE   = '/content/drive/MyDrive/FYP_25-26/runs'
MODEL_PATH_X = os.path.join(DRIVE_BASE, 'best_x.pt')
MODEL_PATH_M = os.path.join(DRIVE_BASE, 'best_m.pt')
FALLBACK_X   = 'yolov8x.pt'
FALLBACK_M   = 'yolov8m.pt'

# ===== ການຕັ້ງຄ່າ =====
CONF_THRESHOLD = 0.25
IMG_SIZE       = 640

try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
except ImportError:
    DEVICE = 'cpu'

EMERGENCY_CLASSES = {'ambulance', 'firetruck', 'police'}

MODEL_INFO = {
    'x': {'name': 'YOLOv8x', 'params': '68.2M parameters'},
    'm': {'name': 'YOLOv8m', 'params': '25.9M parameters'},
}
print(f'[config] DEVICE = {DEVICE}')


In [ ]:
%%writefile detector.py
import time
import threading
from pathlib import Path
import cv2
from ultralytics import YOLO
import config

CLASS_COLORS = {
    'ambulance': (53, 107, 255),
    'firetruck': (53,  53, 224),
    'police':   (255, 158,  75),
}
DEFAULT_COLOR = (122, 196, 0)


class Detector:
    def __init__(self):
        self._models = {}
        self._lock   = threading.Lock()
        self.last_infer_ms = 0.0
        self.using_custom  = False

        x_path = Path(config.MODEL_PATH_X)
        if x_path.exists():
            self._models['x'] = YOLO(str(x_path))
            self.using_custom  = True
            print(f'[Detector] ✅ YOLOv8x custom: {x_path}')
        else:
            self._models['x'] = YOLO(config.FALLBACK_X)
            print(f'[Detector] ⚠️  YOLOv8x: ໃຊ້ pretrained {config.FALLBACK_X}')

        m_path = Path(config.MODEL_PATH_M)
        if m_path.exists():
            self._models['m'] = YOLO(str(m_path))
            self.using_custom  = True
            print(f'[Detector] ✅ YOLOv8m custom: {m_path}')
        else:
            self._models['m'] = YOLO(config.FALLBACK_M)
            print(f'[Detector] ⚠️  YOLOv8m: ໃຊ້ pretrained {config.FALLBACK_M}')

    def get_model(self, key):
        return self._models.get(key, self._models['x'])

    def get_model_info(self, key):
        return config.MODEL_INFO.get(key, config.MODEL_INFO['x'])

    def color_for(self, name):
        return CLASS_COLORS.get(name.lower(), DEFAULT_COLOR)

    def infer(self, frame, model_key='x', conf=None):
        if conf is None:
            conf = config.CONF_THRESHOLD
        model = self.get_model(model_key)
        t0 = time.time()
        with self._lock:
            results = model.predict(
                frame, imgsz=config.IMG_SIZE,
                conf=conf, device=config.DEVICE, verbose=False,
            )
        self.last_infer_ms = (time.time() - t0) * 1000
        dets  = []
        r     = results[0]
        names = model.names
        if r.boxes is not None:
            for box in r.boxes:
                cls_id         = int(box.cls[0])
                conf_val       = float(box.conf[0])
                x1, y1, x2, y2 = map(int, box.xyxy[0])
                dets.append({'name': names.get(cls_id, str(cls_id)),
                             'conf': conf_val, 'box': (x1, y1, x2, y2)})
        return dets

    def draw(self, frame, dets):
        out = frame.copy()
        for d in dets:
            x1, y1, x2, y2 = d['box']
            color = self.color_for(d['name'])
            label = f"{d['name'].upper()} {int(d['conf']*100)}%"
            cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
            (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
            cv2.rectangle(out, (x1, y1-th-8), (x1+tw+8, y1), color, -1)
            cv2.putText(out, label, (x1+4, y1-5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 1, cv2.LINE_AA)
        return out


In [ ]:
%%writefile main.py
import base64, os, tempfile, time
import cv2
import numpy as np
from fastapi import FastAPI, Request, UploadFile, File, Form
from fastapi.responses import JSONResponse
from fastapi.staticfiles import StaticFiles
from fastapi.templating import Jinja2Templates
from detector import Detector

app = FastAPI(title='Emergency Vehicle Detection')
app.mount('/static', StaticFiles(directory='static'), name='static')
templates = Jinja2Templates(directory='templates')

detector = Detector()
print(f'[Server] ✅ ໂມເດລໂຫລດ — custom={detector.using_custom}')


@app.get('/')
def index(request: Request):
    return templates.TemplateResponse(request, 'index.html', {})


@app.post('/predict/image')
async def predict_image(
    file:       UploadFile = File(...),
    model_name: str        = Form('x'),
    conf:       float      = Form(0.25),
):
    data  = await file.read()
    arr   = np.frombuffer(data, dtype=np.uint8)
    frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if frame is None:
        return JSONResponse({'error': 'ບໍ່ສາມາດອ່ານໄຟລ ຮູບພາບ'}, status_code=400)
    dets      = detector.infer(frame, model_key=model_name, conf=conf)
    annotated = detector.draw(frame, dets)
    _, buf    = cv2.imencode('.jpg', annotated, [cv2.IMWRITE_JPEG_QUALITY, 85])
    img_b64   = base64.b64encode(buf.tobytes()).decode()
    info      = detector.get_model_info(model_name)
    return JSONResponse({
        'image':        img_b64,
        'detections':   [{'name': d['name'], 'conf': round(d['conf'],3),
                          'box': list(d['box'])} for d in dets],
        'infer_ms':     round(detector.last_infer_ms, 1),
        'model_name':   info['name'],
        'model_params': info['params'],
    })


@app.post('/predict/video')
async def predict_video(
    file:       UploadFile = File(...),
    model_name: str        = Form('x'),
    conf:       float      = Form(0.25),
):
    data   = await file.read()
    suffix = os.path.splitext(file.filename or '.mp4')[1] or '.mp4'
    with tempfile.NamedTemporaryFile(delete=False, suffix=suffix) as tmp:
        tmp.write(data)
        tmp_path = tmp.name
    summary    = {}
    sample_b64 = None
    try:
        cap         = cv2.VideoCapture(tmp_path)
        frame_count = 0
        t_start     = time.time()
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            dets = detector.infer(frame, model_key=model_name, conf=conf)
            frame_count += 1
            for d in dets:
                n = d['name'].lower()
                summary[n] = summary.get(n, 0) + 1
            if dets and sample_b64 is None:
                ann        = detector.draw(frame, dets)
                _, buf     = cv2.imencode('.jpg', ann, [cv2.IMWRITE_JPEG_QUALITY, 80])
                sample_b64 = base64.b64encode(buf.tobytes()).decode()
        cap.release()
        total_ms = (time.time() - t_start) * 1000
        info     = detector.get_model_info(model_name)
        return JSONResponse({
            'frame_count':        frame_count,
            'total_ms':           round(total_ms, 1),
            'avg_ms_per_frame':   round(total_ms / max(frame_count, 1), 1),
            'detections_summary': summary,
            'sample_frame':       sample_b64,
            'model_name':         info['name'],
            'model_params':       info['params'],
        })
    finally:
        os.unlink(tmp_path)


@app.post('/predict/webcam_frame')
async def predict_webcam_frame(request: Request):
    body       = await request.json()
    img_b64    = body.get('image', '')
    model_name = body.get('model_name', 'm')
    conf       = float(body.get('conf', 0.25))
    raw        = base64.b64decode(img_b64.split(',')[-1])
    arr        = np.frombuffer(raw, dtype=np.uint8)
    frame      = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if frame is None:
        return JSONResponse({'error': 'decode error'}, status_code=400)
    dets = detector.infer(frame, model_key=model_name, conf=conf)
    info = detector.get_model_info(model_name)
    return JSONResponse({
        'detections':   [{'name': d['name'], 'conf': round(d['conf'],3),
                          'box': list(d['box'])} for d in dets],
        'infer_ms':     round(detector.last_infer_ms, 1),
        'model_name':   info['name'],
        'model_params': info['params'],
    })


## 5️⃣  ສ້າງໄຟລ HTML / CSS / JS

In [ ]:
!mkdir -p templates static
print('✅ ສ້າງ directory ສຳເລັດ')


In [ ]:
%%writefile templates/index.html
<!DOCTYPE html>
<html lang='lo'>
<head>
<meta charset='UTF-8'>
<meta name='viewport' content='width=device-width, initial-scale=1.0'>
<title>ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ</title>
<link rel='preconnect' href='https://fonts.googleapis.com'>
<link href='https://fonts.googleapis.com/css2?family=Noto+Sans+Lao:wght@400;600;700&family=Inter:wght@400;500;600;700&display=swap' rel='stylesheet'>
<link rel='stylesheet' href='/static/app.css'>
</head>
<body>

<!-- ===== About Bar ===== -->
<div class='about-bar'>
  <div class='about-left'>
    <div class='about-logo-dot'></div>
    <div>
      <div class='about-uni'>ມະຫາວິທະຍາໄລແຫ່ງຊາດ (NUOL)</div>
      <div class='about-dept'>ຄະນະວິທະຍາສາດທຳມະຊາດ · ວິທະຍາສາດຄອມພິວເຕີ</div>
    </div>
  </div>
  <div class='about-center'>
    <span class='about-sys'>ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ</span>
  </div>
  <div class='about-right'>
    <span class='about-badge' id='modelBadge'>YOLOv8x</span>
    <span class='about-badge fyp'>FYP 2025–26</span>
  </div>
</div>

<!-- ===== Layout ===== -->
<div class='layout'>

  <!-- Sidebar -->
  <aside class='sidebar' id='sidebar'>
    <div class='sidebar-header'>
      <span class='sidebar-brand'>EVD System</span>
      <button class='toggle-btn' id='toggleBtn' title='ພັບ / ຂະຫຍາຍ'>&#9776;</button>
    </div>
    <ul class='nav-list'>
      <li class='nav-item active' data-page='home'>
        <span class='nav-icon'>&#127968;</span>
        <span class='nav-label'>ໜ້າຫຼັກ</span>
      </li>
      <li class='nav-item' data-page='image'>
        <span class='nav-icon'>&#128269;</span>
        <span class='nav-label'>ກວດຈັບຮູບພາບ</span>
      </li>
      <li class='nav-item' data-page='video'>
        <span class='nav-icon'>&#127916;</span>
        <span class='nav-label'>ກວດຈັບວິດີໂອ</span>
      </li>
      <li class='nav-item' data-page='webcam'>
        <span class='nav-icon'>&#128249;</span>
        <span class='nav-label'>ກ້ອງ Real-time</span>
      </li>
      <li class='nav-item' data-page='stats'>
        <span class='nav-icon'>&#128202;</span>
        <span class='nav-label'>ຜົນການທົດສອບ</span>
      </li>
      <li class='nav-item' data-page='about'>
        <span class='nav-icon'>&#8505;&#65039;</span>
        <span class='nav-label'>ກ່ຽວກັບລະບົບ</span>
      </li>
    </ul>
    <div class='sidebar-footer'>
      <span class='nav-label'>v2.0 · YOLOv8</span>
    </div>
  </aside>

  <!-- Main Content -->
  <main class='content' id='content'>

    <!-- ===== HOME ===== -->
    <section class='page active' id='page-home'>
      <div class='hero-section'>
        <div class='hero-eyebrow'>Real-time · YOLOv8 · Emergency Detection</div>
        <h1 class='hero-title-lao'>ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ</h1>
        <p class='hero-title-en'>Emergency Vehicle Detection System</p>
        <p class='hero-desc'>ກວດຈັບຍານພາຫະນະສຸກເສີນດ້ວຍ YOLOv8 ໃນເວລາຈິງ · mAP50 = 97.54%</p>
      </div>
      <div class='vehicle-cards'>
        <div class='vehicle-card amb' onclick='goPage("image")'>
          <div class='vehicle-icon'>&#128657;</div>
          <div class='vehicle-name'>ລົດໂຮງໝໍ</div>
          <div class='vehicle-en'>Ambulance</div>
        </div>
        <div class='vehicle-card fire' onclick='goPage("image")'>
          <div class='vehicle-icon'>&#128658;</div>
          <div class='vehicle-name'>ລົດດັບເພີງ</div>
          <div class='vehicle-en'>Fire Truck</div>
        </div>
        <div class='vehicle-card police' onclick='goPage("image")'>
          <div class='vehicle-icon'>&#128660;</div>
          <div class='vehicle-name'>ລົດຕຳຫຼວດ</div>
          <div class='vehicle-en'>Police Car</div>
        </div>
      </div>
      <div class='home-actions'>
        <button class='btn btn-primary' onclick='goPage("image")'>&#128269; ກວດຈັບຮູບພາບ</button>
        <button class='btn btn-outline' onclick='goPage("webcam")'>&#128249; ເປີດກ້ອງ Real-time</button>
        <button class='btn btn-outline' onclick='goPage("stats")'>&#128202; ຜົນການທົດສອບ</button>
      </div>
    </section>

    <!-- ===== IMAGE ===== -->
    <section class='page' id='page-image'>
      <div class='page-header'>
        <h2 class='page-title'>&#128269; ກວດຈັບຮູບພາບ</h2>
        <p class='page-subtitle'>ອັບໂຫລດຮູບພາບ ເພື່ອກວດຈັບຍານພາຫະນະສຸກເສີນ</p>
      </div>
      <div class='controls-bar'>
        <div class='ctrl-group'>
          <label class='ctrl-label'>ແບບຈຳລອງ</label>
          <select id='imgModel' class='ctrl-select'>
            <option value='x'>YOLOv8x · ຖືກຕ້ອງກວ່າ (68.2M params)</option>
            <option value='m' selected>YOLOv8m · ໄວກວ່າ (25.9M params)</option>
          </select>
        </div>
        <div class='ctrl-group'>
          <label class='ctrl-label'>Confidence: <span class='conf-badge' id='imgConfVal'>0.25</span></label>
          <input type='range' id='imgConf' min='0' max='1' step='0.01' value='0.25' class='ctrl-slider'>
        </div>
      </div>
      <div class='upload-zone' id='imgDrop'>
        <input type='file' id='imgFile' accept='image/*' hidden>
        <div class='upload-icon'>&#128444;&#65039;</div>
        <div class='upload-text'>ຄລິກ ຫຼື ລາກຮູບພາບ ມາໃສ່ບ່ອນນີ້</div>
        <div class='upload-hint'>ຮອງຮັບ: JPG · PNG · WEBP</div>
      </div>
      <div style='margin-top:14px;display:flex;gap:10px'>
        <button class='btn btn-primary' id='imgBtn' disabled>ກວດຈັບ</button>
      </div>
      <div id='imgResult' class='result-box hidden'>
        <div class='result-meta' id='imgMeta'></div>
        <div class='result-layout'>
          <img id='imgOut' class='result-img' alt='ຜົນການກວດຈັບ'>
          <div class='det-list' id='imgDetList'></div>
        </div>
      </div>
    </section>

    <!-- ===== VIDEO ===== -->
    <section class='page' id='page-video'>
      <div class='page-header'>
        <h2 class='page-title'>&#127916; ກວດຈັບວິດີໂອ</h2>
        <p class='page-subtitle'>ອັບໂຫລດວິດີໂອ ລະບົບຈະປະມວນຜົນທຸກ frame</p>
      </div>
      <div class='controls-bar'>
        <div class='ctrl-group'>
          <label class='ctrl-label'>ແບບຈຳລອງ</label>
          <select id='vidModel' class='ctrl-select'>
            <option value='x'>YOLOv8x · ຖືກຕ້ອງກວ່າ (68.2M params)</option>
            <option value='m' selected>YOLOv8m · ໄວກວ່າ (25.9M params)</option>
          </select>
        </div>
        <div class='ctrl-group'>
          <label class='ctrl-label'>Confidence: <span class='conf-badge' id='vidConfVal'>0.25</span></label>
          <input type='range' id='vidConf' min='0' max='1' step='0.01' value='0.25' class='ctrl-slider'>
        </div>
      </div>
      <div class='upload-zone' id='vidDrop'>
        <input type='file' id='vidFile' accept='video/*' hidden>
        <div class='upload-icon'>&#127916;</div>
        <div class='upload-text'>ຄລິກ ຫຼື ລາກໄຟລ ວິດີໂອ ມາໃສ່ບ່ອນນີ້</div>
        <div class='upload-hint'>ຮອງຮັບ: MP4 · AVI · MOV</div>
      </div>
      <div style='margin-top:14px;display:flex;gap:10px'>
        <button class='btn btn-primary' id='vidBtn' disabled>ປະມວນຜົນວິດີໂອ</button>
      </div>
      <div id='vidProgress' class='progress-row hidden'>
        <div class='spinner'></div>
        <span id='vidProgText'>ກຳລັງປະມວນຜົນ — ກະລຸນາລໍຖ້າ...</span>
      </div>
      <div id='vidResult' class='result-box hidden'>
        <div class='result-meta' id='vidMeta'></div>
        <div class='vstat-grid' id='vidStats'></div>
        <div id='vidSampleWrap' class='hidden'>
          <div class='sample-label'>ຕົວຢ່າງ frame ທີ່ພົບຍານພາຫະນະສຸກເສີນ</div>
          <img id='vidSample' class='result-img' alt='ຕົວຢ່າງ frame'>
        </div>
      </div>
    </section>

    <!-- ===== WEBCAM ===== -->
    <section class='page' id='page-webcam'>
      <div class='page-header'>
        <h2 class='page-title'>&#128249; ກ້ອງ Real-time</h2>
        <p class='page-subtitle'>ກວດຈັບຍານພາຫະນະສຸກເສີນຜ່ານ Webcam ໃນເວລາຈິງ</p>
      </div>
      <div class='controls-bar'>
        <div class='ctrl-group'>
          <label class='ctrl-label'>ແບບຈຳລອງ</label>
          <select id='camModel' class='ctrl-select'>
            <option value='x'>YOLOv8x · ຖືກຕ້ອງກວ່າ (68.2M params)</option>
            <option value='m' selected>YOLOv8m · ໄວກວ່າ (25.9M params)</option>
          </select>
        </div>
        <div class='ctrl-group'>
          <label class='ctrl-label'>Confidence: <span class='conf-badge' id='camConfVal'>0.25</span></label>
          <input type='range' id='camConf' min='0' max='1' step='0.01' value='0.25' class='ctrl-slider'>
        </div>
      </div>
      <div class='webcam-container'>
        <div class='webcam-view'>
          <video id='camVideo' autoplay muted playsinline></video>
          <canvas id='camCanvas'></canvas>
          <div class='wcam-fps' id='camFps'>-- FPS</div>
          <div class='wcam-model' id='camModelInfo'>--</div>
        </div>
        <div style='display:flex;gap:10px;margin-bottom:12px'>
          <button class='btn btn-primary' id='camStart'>&#9654; ເລີ່ມກ້ອງ</button>
          <button class='btn btn-danger' id='camStop' disabled>&#9632; ຢຸດກ້ອງ</button>
        </div>
        <div id='camError' class='error-msg hidden'></div>
        <div id='camHint' class='hint-msg hidden'>
          &#128161; FPS ຕ່ຳ? ລອງໃຊ້ <strong>YOLOv8m</strong> ຫຼື ເພີ່ມ Confidence threshold
        </div>
      </div>
    </section>

    <!-- ===== STATS ===== -->
    <section class='page' id='page-stats'>
      <div class='page-header'>
        <h2 class='page-title'>&#128202; ຜົນການທົດສອບ</h2>
        <p class='page-subtitle'>ຜົນການທົດສອບໂມເດລ YOLOv8 ສຳລັບ Dataset ຍານພາຫະນະສຸກເສີນ</p>
      </div>
      <div class='metrics-grid'>
        <div class='metric-card'>
          <div class='metric-val map'>97.54%</div>
          <div class='metric-lbl'>mAP50</div>
          <div class='metric-sub'>Mean Average Precision @IoU 0.5</div>
        </div>
        <div class='metric-card'>
          <div class='metric-val prec'>95.03%</div>
          <div class='metric-lbl'>Precision</div>
          <div class='metric-sub'>ຄວາມຖືກຕ້ອງ (True Positive Rate)</div>
        </div>
        <div class='metric-card'>
          <div class='metric-val rec'>96.00%</div>
          <div class='metric-lbl'>Recall</div>
          <div class='metric-sub'>ຄວາມຄົບຖ້ວນ (Sensitivity)</div>
        </div>
      </div>
      <div class='section-card'>
        <div class='section-title'>ຜົນລາຍ Class</div>
        <table class='class-table'>
          <thead>
            <tr>
              <th>Class</th>
              <th>mAP50</th>
              <th>Precision</th>
              <th>Recall</th>
              <th>ຄ່ວາມຖືກຕ້ອງ</th>
            </tr>
          </thead>
          <tbody>
            <tr>
              <td><span class='cls-dot amb'></span>ລົດໂຮງໝໍ (Ambulance)</td>
              <td><strong>98.1%</strong></td>
              <td>96.2%</td>
              <td>97.0%</td>
              <td><div class='bar-wrap'><div class='bar-fill' style='width:98.1%;background:#ea580c'></div></div></td>
            </tr>
            <tr>
              <td><span class='cls-dot fire'></span>ລົດດັບເພີງ (Fire Truck)</td>
              <td><strong>97.3%</strong></td>
              <td>94.5%</td>
              <td>95.8%</td>
              <td><div class='bar-wrap'><div class='bar-fill' style='width:97.3%;background:#dc2626'></div></div></td>
            </tr>
            <tr>
              <td><span class='cls-dot pol'></span>ລົດຕຳຫຼວດ (Police)</td>
              <td><strong>97.2%</strong></td>
              <td>94.4%</td>
              <td>95.2%</td>
              <td><div class='bar-wrap'><div class='bar-fill' style='width:97.2%;background:#2563eb'></div></div></td>
            </tr>
          </tbody>
        </table>
      </div>
      <div class='section-card'>
        <div class='section-title'>ປຽບທຽບໂມເດລ</div>
        <div class='model-compare'>
          <div class='model-info-card'>
            <div class='model-badge-lg x'>YOLOv8x</div>
            <div class='model-detail'>68.2M parameters</div>
            <div class='model-detail'>&#128994; ຄວາມຖືກຕ້ອງສູງສຸດ</div>
            <div class='model-detail'>GPU ≥ 8 GB VRAM</div>
          </div>
          <div class='model-info-card'>
            <div class='model-badge-lg m'>YOLOv8m</div>
            <div class='model-detail'>25.9M parameters</div>
            <div class='model-detail'>&#9889; ສົມດຸນ ໄວ + ຖືກ</div>
            <div class='model-detail'>GPU ທົ່ວໄປ</div>
          </div>
        </div>
      </div>
    </section>

    <!-- ===== ABOUT ===== -->
    <section class='page' id='page-about'>
      <div class='page-header'>
        <h2 class='page-title'>&#8505;&#65039; ກ່ຽວກັບລະບົບ</h2>
        <p class='page-subtitle'>ຂໍ້ມູນໂຄງການ ແລະ ເຕັກໂນໂລຈີທີ່ໃຊ້</p>
      </div>
      <div class='about-section'>
        <div class='about-card'>
          <div class='about-card-title'>&#128203; ຂໍ້ມູນໂຄງການ</div>
          <div class='about-item'><span class='about-key'>ຊື່ລະບົບ (ລາວ):</span><span class='about-val'>ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ</span></div>
          <div class='about-item'><span class='about-key'>ຊື່ລະບົບ (Eng):</span><span class='about-val'>Emergency Vehicle Detection System</span></div>
          <div class='about-item'><span class='about-key'>ໂຄງການ:</span><span class='about-val'>Final Year Project (FYP) 2025–2026</span></div>
          <div class='about-item'><span class='about-key'>ມະຫາວິທະຍາໄລ:</span><span class='about-val'>ມະຫາວິທະຍາໄລແຫ່ງຊາດ (NUOL)</span></div>
          <div class='about-item'><span class='about-key'>ຄະນະ:</span><span class='about-val'>ຄະນະວິທະຍາສາດທຳມະຊາດ · ວິທະຍາສາດຄອມພິວເຕີ</span></div>
        </div>
        <div class='about-card'>
          <div class='about-card-title'>&#128736;&#65039; ເຕັກໂນໂລຈີ</div>
          <div class='about-item'><span class='about-key'>AI Model:</span><span class='about-val'>YOLOv8x / YOLOv8m (Ultralytics)</span></div>
          <div class='about-item'><span class='about-key'>Backend:</span><span class='about-val'>FastAPI + Python 3.10 + Uvicorn</span></div>
          <div class='about-item'><span class='about-key'>Computer Vision:</span><span class='about-val'>OpenCV (opencv-python-headless)</span></div>
          <div class='about-item'><span class='about-key'>Tunnel:</span><span class='about-val'>pyngrok (Ngrok)</span></div>
          <div class='about-item'><span class='about-key'>Training GPU:</span><span class='about-val'>Google Colab Pro · A100</span></div>
          <div class='about-item'><span class='about-key'>Classes:</span><span class='about-val'>Ambulance · Fire Truck · Police Car</span></div>
        </div>
        <div class='about-card'>
          <div class='about-card-title'>&#128202; ສະຫຼຸບຜົນ</div>
          <div class='about-item'><span class='about-key'>mAP50:</span><span class='about-val' style='color:#1A6BFF;font-size:18px;font-weight:700'>97.54%</span></div>
          <div class='about-item'><span class='about-key'>Precision:</span><span class='about-val' style='color:#16a34a;font-size:18px;font-weight:700'>95.03%</span></div>
          <div class='about-item'><span class='about-key'>Recall:</span><span class='about-val' style='color:#d97706;font-size:18px;font-weight:700'>96.00%</span></div>
        </div>
      </div>
    </section>

  </main>
</div>

<script src='/static/app.js'></script>
</body>
</html>


In [ ]:
%%writefile static/app.css
@import url('https://fonts.googleapis.com/css2?family=Noto+Sans+Lao:wght@400;600;700&family=Inter:wght@400;500;600;700&display=swap');

:root {
  --about-h:   48px;
  --sidebar-w: 220px;
  --sidebar-c: 60px;
  --blue:   #1A6BFF;
  --green:  #16a34a;
  --red:    #dc2626;
  --amber:  #d97706;
  --text:   #0f172a;
  --muted:  #6b7280;
  --border: #e2e6ed;
  --bg:     #f5f7fa;
  --white:  #ffffff;
  --card:   #f8fafc;
  --shadow: 0 1px 4px rgba(0,0,0,0.08);
  --trans:  0.25s ease;
}

* { box-sizing: border-box; margin: 0; padding: 0; }
html, body { height: 100%; }
body {
  font-family: 'Noto Sans Lao', 'Inter', 'Segoe UI', system-ui, sans-serif;
  background: var(--bg); color: var(--text);
}

/* ===== About Bar ===== */
.about-bar {
  position: fixed; top: 0; left: 0; right: 0;
  height: var(--about-h); z-index: 300;
  background: var(--white); border-bottom: 1px solid var(--border);
  display: flex; align-items: center; justify-content: space-between;
  padding: 0 20px; box-shadow: var(--shadow);
  gap: 12px;
}
.about-left  { display: flex; align-items: center; gap: 10px; min-width: 0; }
.about-center { flex: 1; text-align: center; }
.about-right { display: flex; align-items: center; gap: 8px; flex-shrink: 0; }

.about-logo-dot {
  width: 10px; height: 10px; border-radius: 50%;
  background: var(--green); box-shadow: 0 0 8px var(--green);
  animation: pulse 2s infinite; flex-shrink: 0;
}
.about-uni  { font-size: 12px; font-weight: 700; color: var(--text); white-space: nowrap; }
.about-dept { font-size: 10px; color: var(--muted); white-space: nowrap; }
.about-sys  { font-size: 13px; font-weight: 700; color: var(--blue); letter-spacing: 0.02em; }
.about-badge {
  font-size: 11px; font-weight: 600; padding: 3px 10px;
  border-radius: 20px; background: #eff6ff; color: var(--blue);
  border: 1px solid #bfdbfe; white-space: nowrap;
}
.about-badge.fyp { background: #f0fdf4; color: var(--green); border-color: #bbf7d0; }

/* ===== Layout ===== */
.layout {
  display: flex;
  padding-top: var(--about-h);
  min-height: 100vh;
}

/* ===== Sidebar ===== */
.sidebar {
  position: fixed; top: var(--about-h); left: 0; bottom: 0;
  width: var(--sidebar-w);
  background: var(--white); border-right: 1px solid var(--border);
  z-index: 200; display: flex; flex-direction: column;
  transition: width var(--trans); overflow: hidden;
  box-shadow: 2px 0 8px rgba(0,0,0,0.04);
}
.sidebar.collapsed { width: var(--sidebar-c); }

.sidebar-header {
  display: flex; align-items: center; justify-content: space-between;
  padding: 0 14px; height: 52px;
  border-bottom: 1px solid var(--border); flex-shrink: 0;
}
.sidebar-brand {
  font-size: 13px; font-weight: 700; color: var(--blue);
  white-space: nowrap; overflow: hidden;
  transition: opacity var(--trans), width var(--trans);
}
.sidebar.collapsed .sidebar-brand { opacity: 0; width: 0; }

.toggle-btn {
  background: none; border: none; cursor: pointer;
  color: var(--muted); font-size: 18px;
  border-radius: 6px; padding: 4px;
  display: flex; align-items: center; justify-content: center;
  width: 32px; height: 32px; flex-shrink: 0;
  transition: background 0.15s, color 0.15s;
}
.toggle-btn:hover { background: var(--bg); color: var(--blue); }

.nav-list { list-style: none; padding: 8px 0; flex: 1; overflow-y: auto; }

.nav-item {
  display: flex; align-items: center; gap: 12px;
  padding: 11px 16px; cursor: pointer;
  font-size: 13px; font-weight: 500; color: var(--muted);
  border-left: 3px solid transparent;
  transition: all 0.15s; white-space: nowrap;
  user-select: none;
}
.nav-item:hover { background: #f0f4ff; color: var(--blue); }
.nav-item.active {
  background: #eff6ff; color: var(--blue);
  border-left-color: var(--blue); font-weight: 600;
}
.nav-icon  { font-size: 18px; flex-shrink: 0; width: 24px; text-align: center; line-height: 1; }
.nav-label { overflow: hidden; transition: opacity var(--trans), width var(--trans); }
.sidebar.collapsed .nav-label   { opacity: 0; width: 0; }
.sidebar.collapsed .nav-item    { padding: 11px 0; justify-content: center; }
.sidebar.collapsed .nav-icon    { width: auto; }

.sidebar-footer {
  padding: 10px 14px; border-top: 1px solid var(--border);
  font-size: 10px; color: var(--muted); flex-shrink: 0;
  overflow: hidden;
}
.sidebar.collapsed .sidebar-footer { opacity: 0; }

/* ===== Content ===== */
.content {
  margin-left: var(--sidebar-w);
  flex: 1; min-height: calc(100vh - var(--about-h));
  transition: margin-left var(--trans);
  padding: 30px 36px;
}
.content.shifted { margin-left: var(--sidebar-c); }

/* ===== Pages ===== */
.page { display: none; max-width: 900px; }
.page.active { display: block; }

/* ===== HOME ===== */
.hero-section { text-align: center; padding: 48px 20px 36px; }
.hero-eyebrow {
  font-size: 11px; font-weight: 600; letter-spacing: 0.1em;
  color: var(--blue); text-transform: uppercase; margin-bottom: 16px;
}
.hero-title-lao {
  font-size: 36px; font-weight: 700; color: var(--text);
  line-height: 1.3; margin-bottom: 10px;
}
.hero-title-en {
  font-size: 17px; font-weight: 500; color: var(--muted);
  margin-bottom: 12px;
}
.hero-desc {
  font-size: 14px; color: var(--muted); max-width: 560px;
  margin: 0 auto 32px; line-height: 1.7;
}

.vehicle-cards {
  display: flex; gap: 20px; justify-content: center;
  flex-wrap: wrap; margin-bottom: 28px;
}
.vehicle-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: 16px; padding: 28px 20px;
  text-align: center; flex: 1 1 160px; max-width: 200px;
  cursor: pointer; transition: transform 0.2s, box-shadow 0.2s;
  box-shadow: var(--shadow);
}
.vehicle-card:hover { transform: translateY(-4px); box-shadow: 0 8px 24px rgba(26,107,255,0.12); }
.vehicle-card.amb   { border-top: 3px solid #ea580c; }
.vehicle-card.fire  { border-top: 3px solid var(--red); }
.vehicle-card.police { border-top: 3px solid #2563eb; }
.vehicle-icon { font-size: 48px; margin-bottom: 10px; line-height: 1; }
.vehicle-name { font-size: 15px; font-weight: 700; color: var(--text); margin-bottom: 4px; }
.vehicle-en   { font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.06em; }

.home-actions { display: flex; gap: 12px; justify-content: center; flex-wrap: wrap; }

/* ===== Page Header ===== */
.page-header { margin-bottom: 22px; }
.page-title   { font-size: 22px; font-weight: 700; margin-bottom: 4px; }
.page-subtitle { font-size: 13px; color: var(--muted); }

/* ===== Controls Bar ===== */
.controls-bar {
  display: flex; gap: 20px; align-items: center;
  background: var(--white); border: 1px solid var(--border);
  border-radius: 12px; padding: 14px 18px; margin-bottom: 18px;
  flex-wrap: wrap; box-shadow: var(--shadow);
}
.ctrl-group { display: flex; align-items: center; gap: 10px; }
.ctrl-label {
  font-size: 12px; font-weight: 600; color: var(--muted);
  letter-spacing: 0.04em; white-space: nowrap;
}
.ctrl-select {
  font-size: 12px; padding: 6px 10px; border-radius: 8px;
  border: 1px solid var(--border); background: var(--bg); color: var(--text);
  cursor: pointer; min-width: 220px;
}
.ctrl-slider { width: 140px; cursor: pointer; accent-color: var(--blue); }
.conf-badge {
  min-width: 36px; text-align: center; font-family: monospace;
  font-size: 13px; font-weight: 700; color: var(--blue);
}

/* ===== Buttons ===== */
.btn {
  padding: 10px 22px; border-radius: 8px; font-size: 13px;
  font-weight: 600; border: none; cursor: pointer; transition: all 0.15s;
  font-family: inherit;
}
.btn-primary { background: var(--blue); color: #fff; }
.btn-primary:hover { opacity: 0.88; }
.btn-primary:disabled { opacity: 0.4; cursor: not-allowed; }
.btn-danger  { background: var(--red); color: #fff; }
.btn-danger:hover  { opacity: 0.88; }
.btn-danger:disabled { opacity: 0.4; cursor: not-allowed; }
.btn-outline {
  background: var(--white); color: var(--blue);
  border: 1.5px solid var(--blue);
}
.btn-outline:hover { background: #eff6ff; }

/* ===== Upload Zone ===== */
.upload-zone {
  border: 2px dashed var(--border); border-radius: 12px;
  padding: 48px 20px; text-align: center; cursor: pointer;
  background: var(--white); transition: all 0.2s;
}
.upload-zone:hover, .upload-zone.drag-over { border-color: var(--blue); background: #f0f6ff; }
.upload-zone.has-file { border-color: var(--green); background: #f0fdf4; }
.upload-icon { font-size: 44px; margin-bottom: 10px; line-height: 1; }
.upload-text { font-size: 15px; font-weight: 600; margin-bottom: 6px; }
.upload-hint { font-size: 12px; color: var(--muted); }

/* ===== Progress ===== */
.progress-row {
  display: flex; align-items: center; gap: 10px;
  font-size: 13px; color: var(--muted); padding: 10px 0;
}
.progress-row.hidden { display: none; }
.spinner {
  width: 18px; height: 18px; border-radius: 50%;
  border: 2px solid var(--border); border-top-color: var(--blue);
  animation: spin 0.7s linear infinite; flex-shrink: 0;
}
@keyframes spin { to { transform: rotate(360deg); } }
@keyframes pulse { 0%,100% { opacity: 1; } 50% { opacity: 0.35; } }

/* ===== Result ===== */
.result-box { margin-top: 20px; }
.result-box.hidden { display: none; }
.result-meta {
  font-size: 12px; color: var(--muted);
  background: var(--white); border: 1px solid var(--border);
  border-radius: 8px; padding: 10px 16px; margin-bottom: 14px; line-height: 1.9;
}
.result-meta strong { color: var(--blue); }
.result-layout { display: flex; gap: 16px; flex-wrap: wrap; }
.result-img {
  max-width: 100%; border-radius: 10px; border: 1px solid var(--border);
  flex: 1 1 380px;
}
.det-list { flex: 0 0 200px; display: flex; flex-direction: column; gap: 6px; }
.det-item {
  padding: 10px 12px; border-radius: 8px; font-size: 12px;
  background: var(--white); border: 1px solid var(--border);
  border-left: 3px solid var(--muted);
}
.det-item.ambulance { border-left-color: #ea580c; }
.det-item.firetruck { border-left-color: var(--red); }
.det-item.police    { border-left-color: #2563eb; }
.det-name { font-weight: 700; text-transform: capitalize; }
.det-conf { color: var(--muted); font-size: 11px; margin-top: 2px; }

/* ===== Video Stats ===== */
.vstat-grid { display: flex; gap: 12px; flex-wrap: wrap; margin-bottom: 16px; }
.vstat-card {
  flex: 1 1 120px; background: var(--white); border: 1px solid var(--border);
  border-radius: 10px; padding: 14px 16px; text-align: center;
}
.vstat-num { font-size: 24px; font-weight: 700; color: var(--blue); font-family: monospace; }
.vstat-lbl { font-size: 10px; color: var(--muted); margin-top: 4px; letter-spacing: 0.06em; text-transform: uppercase; }
.sample-label {
  font-size: 11px; font-weight: 600; color: var(--muted);
  margin-bottom: 8px; text-transform: uppercase; letter-spacing: 0.06em;
}

/* ===== Webcam ===== */
.webcam-container { max-width: 780px; }
.webcam-view {
  position: relative; background: #0f1117; border-radius: 12px;
  overflow: hidden; aspect-ratio: 16/9; margin-bottom: 14px;
}
.webcam-view video, .webcam-view canvas {
  position: absolute; inset: 0; width: 100%; height: 100%;
}
.webcam-view canvas { pointer-events: none; }
.wcam-fps {
  position: absolute; top: 10px; left: 10px; z-index: 10;
  font-family: monospace; font-size: 14px; font-weight: 700;
  padding: 4px 10px; border-radius: 6px;
  background: rgba(0,0,0,0.6); color: #4ade80;
}
.wcam-model {
  position: absolute; top: 10px; right: 10px; z-index: 10;
  font-family: monospace; font-size: 11px;
  padding: 4px 10px; border-radius: 6px;
  background: rgba(0,0,0,0.6); color: #93c5fd;
}
.error-msg {
  padding: 12px 16px; border-radius: 8px; font-size: 13px;
  background: #fef2f2; border: 1px solid #fecaca; color: var(--red);
}
.hint-msg {
  padding: 12px 16px; border-radius: 8px; font-size: 13px;
  background: #fffbeb; border: 1px solid #fde68a; color: #92400e;
}
.error-msg.hidden, .hint-msg.hidden { display: none !important; }

/* ===== Stats Page ===== */
.metrics-grid {
  display: flex; gap: 20px; justify-content: flex-start;
  flex-wrap: wrap; margin-bottom: 28px;
}
.metric-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: 16px; padding: 24px 28px;
  text-align: center; flex: 1 1 160px; max-width: 220px;
  box-shadow: var(--shadow);
}
.metric-val { font-size: 34px; font-weight: 700; font-family: monospace; margin-bottom: 6px; }
.metric-val.map  { color: var(--blue); }
.metric-val.prec { color: var(--green); }
.metric-val.rec  { color: var(--amber); }
.metric-lbl { font-size: 12px; font-weight: 700; color: var(--muted); letter-spacing: 0.08em; text-transform: uppercase; }
.metric-sub { font-size: 10px; color: var(--muted); margin-top: 4px; }

.section-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: 12px; padding: 20px 22px; margin-bottom: 16px;
}
.section-title {
  font-size: 13px; font-weight: 700; color: var(--text);
  margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid var(--border);
}
.class-table { width: 100%; border-collapse: collapse; font-size: 13px; }
.class-table th {
  text-align: left; padding: 9px 14px;
  background: var(--bg); color: var(--muted);
  font-size: 10px; font-weight: 600; letter-spacing: 0.06em; text-transform: uppercase;
  border-bottom: 2px solid var(--border);
}
.class-table td { padding: 12px 14px; border-bottom: 1px solid var(--border); vertical-align: middle; }
.class-table tr:last-child td { border-bottom: none; }
.class-table tr:hover td { background: var(--card); }
.cls-dot {
  display: inline-block; width: 10px; height: 10px;
  border-radius: 50%; margin-right: 8px; vertical-align: middle;
}
.cls-dot.amb  { background: #ea580c; }
.cls-dot.fire { background: var(--red); }
.cls-dot.pol  { background: #2563eb; }
.bar-wrap { background: #f1f5f9; border-radius: 4px; height: 8px; width: 120px; }
.bar-fill { height: 8px; border-radius: 4px; }

.model-compare { display: flex; gap: 16px; flex-wrap: wrap; }
.model-info-card {
  flex: 1 1 200px; background: var(--bg); border: 1px solid var(--border);
  border-radius: 10px; padding: 16px; text-align: center;
}
.model-badge-lg {
  display: inline-block; font-size: 14px; font-weight: 700;
  padding: 5px 14px; border-radius: 20px; margin-bottom: 10px;
  font-family: monospace;
}
.model-badge-lg.x { background: #eff6ff; color: var(--blue); border: 1px solid #bfdbfe; }
.model-badge-lg.m { background: #f0fdf4; color: var(--green); border: 1px solid #bbf7d0; }
.model-detail { font-size: 12px; color: var(--muted); margin-top: 4px; }

/* ===== About Page ===== */
.about-section { max-width: 700px; }
.about-card {
  background: var(--white); border: 1px solid var(--border);
  border-radius: 12px; padding: 20px 22px; margin-bottom: 14px;
}
.about-card-title {
  font-size: 14px; font-weight: 700; color: var(--text);
  margin-bottom: 14px; padding-bottom: 10px; border-bottom: 1px solid var(--border);
}
.about-item { display: flex; gap: 12px; padding: 5px 0; font-size: 13px; align-items: baseline; }
.about-key  { color: var(--muted); min-width: 160px; flex-shrink: 0; font-weight: 500; }
.about-val  { color: var(--text); font-weight: 600; }

/* ===== Utilities ===== */
.hidden { display: none !important; }

@media (max-width: 768px) {
  .sidebar { width: var(--sidebar-c); }
  .sidebar .nav-label, .sidebar .sidebar-brand, .sidebar .sidebar-footer { opacity: 0; width: 0; overflow: hidden; }
  .sidebar .nav-item { padding: 11px 0; justify-content: center; }
  .content { margin-left: var(--sidebar-c); padding: 20px 14px; }
  .about-center { display: none; }
  .hero-title-lao { font-size: 24px; }
  .metrics-grid { flex-direction: column; }
}


In [ ]:
%%writefile static/app.js
// ===== Sidebar Toggle =====
const sidebar = document.getElementById('sidebar');
const content = document.getElementById('content');
const toggleBtn = document.getElementById('toggleBtn');

toggleBtn.addEventListener('click', () => {
  sidebar.classList.toggle('collapsed');
  content.classList.toggle('shifted');
});

// ===== Page Navigation =====
function goPage(name) {
  document.querySelectorAll('.page').forEach(p => p.classList.remove('active'));
  document.querySelectorAll('.nav-item').forEach(n => n.classList.remove('active'));
  const page = document.getElementById('page-' + name);
  const nav  = document.querySelector('[data-page="' + name + '"]');
  if (page) page.classList.add('active');
  if (nav)  nav.classList.add('active');
  if (name !== 'webcam' && webcamRunning) stopCam();
}

document.querySelectorAll('.nav-item').forEach(item => {
  item.addEventListener('click', () => goPage(item.dataset.page));
});

// ===== Helpers =====
const DET_COLORS = { ambulance: '#ea580c', firetruck: '#dc2626', police: '#2563eb' };
function detColor(name) { return DET_COLORS[name.toLowerCase()] || '#16a34a'; }

function makeDropZone(zoneId, inputId, onFile) {
  const zone  = document.getElementById(zoneId);
  const input = document.getElementById(inputId);
  zone.addEventListener('click', () => input.click());
  input.addEventListener('change', () => { if (input.files[0]) { zone.classList.add('has-file'); zone.querySelector('.upload-text').textContent = input.files[0].name; onFile(input.files[0]); } });
  zone.addEventListener('dragover',  e => { e.preventDefault(); zone.classList.add('drag-over'); });
  zone.addEventListener('dragleave', () => zone.classList.remove('drag-over'));
  zone.addEventListener('drop', e => {
    e.preventDefault(); zone.classList.remove('drag-over');
    const f = e.dataTransfer.files[0];
    if (f) { zone.classList.add('has-file'); zone.querySelector('.upload-text').textContent = f.name; onFile(f); }
  });
}

function updateModelBadge(name) {
  document.getElementById('modelBadge').textContent = name || 'YOLOv8';
}

// ===== Confidence sliders =====
['imgConf','vidConf','camConf'].forEach(id => {
  const el  = document.getElementById(id);
  const val = document.getElementById(id.replace('Conf','ConfVal'));
  if (el && val) el.addEventListener('input', () => { val.textContent = parseFloat(el.value).toFixed(2); });
});

// ===== IMAGE Detection =====
let imgFile = null;
const imgBtn    = document.getElementById('imgBtn');
const imgResult = document.getElementById('imgResult');
const imgMeta   = document.getElementById('imgMeta');
const imgOut    = document.getElementById('imgOut');
const imgDetList = document.getElementById('imgDetList');

makeDropZone('imgDrop', 'imgFile', f => {
  imgFile = f;
  imgBtn.disabled = false;
  imgResult.classList.add('hidden');
});

imgBtn.addEventListener('click', async () => {
  if (!imgFile) return;
  imgBtn.disabled = true;
  imgBtn.textContent = 'ກຳລັງກວດຈັບ...';
  imgResult.classList.add('hidden');

  const model = document.getElementById('imgModel').value;
  const conf  = document.getElementById('imgConf').value;
  const fd = new FormData();
  fd.append('file', imgFile);
  fd.append('model_name', model);
  fd.append('conf', conf);

  try {
    const res  = await fetch('/predict/image', { method: 'POST', body: fd });
    const data = await res.json();
    if (data.error) { alert(data.error); return; }

    updateModelBadge(data.model_name);
    imgOut.src = 'data:image/jpeg;base64,' + data.image;
    imgMeta.innerHTML =
      `ໂມເດລ: <strong>${data.model_name}</strong> &nbsp;·&nbsp; ${data.model_params} &nbsp;·&nbsp; ` +
      `ປະມວນຜົນ <strong>${data.infer_ms} ms</strong> &nbsp;·&nbsp; ` +
      `ພົບ <strong>${data.detections.length}</strong> ຍານພາຫະນະ`;

    imgDetList.innerHTML = data.detections.length === 0
      ? '<div class="det-item">ບໍ່ພົບຍານພາຫະນະສຸກເສີນ</div>'
      : data.detections.map(d => `
          <div class="det-item ${d.name.toLowerCase()}">
            <div class="det-name">${d.name}</div>
            <div class="det-conf">${Math.round(d.conf*100)}% confidence</div>
          </div>`).join('');

    imgResult.classList.remove('hidden');
  } catch(e) { alert('ຜິດພາດ: ' + e.message); }
  finally {
    imgBtn.disabled = false;
    imgBtn.textContent = 'ກວດຈັບ';
  }
});

// ===== VIDEO Detection =====
let vidFile = null;
const vidBtn        = document.getElementById('vidBtn');
const vidProgress   = document.getElementById('vidProgress');
const vidProgText   = document.getElementById('vidProgText');
const vidResult     = document.getElementById('vidResult');
const vidMeta       = document.getElementById('vidMeta');
const vidStats      = document.getElementById('vidStats');
const vidSampleWrap = document.getElementById('vidSampleWrap');
const vidSample     = document.getElementById('vidSample');

makeDropZone('vidDrop', 'vidFile', f => {
  vidFile = f;
  vidBtn.disabled = false;
  vidResult.classList.add('hidden');
});

vidBtn.addEventListener('click', async () => {
  if (!vidFile) return;
  vidBtn.disabled = true;
  vidBtn.textContent = 'ກຳລັງປະມວນຜົນ...';
  vidResult.classList.add('hidden');
  vidProgress.classList.remove('hidden');
  vidProgText.textContent = 'ກຳລັງສົ່ງ ແລະ ປະມວນຜົນວິດີໂອ — ກະລຸນາລໍຖ້າ...';

  const model = document.getElementById('vidModel').value;
  const conf  = document.getElementById('vidConf').value;
  const fd = new FormData();
  fd.append('file', vidFile);
  fd.append('model_name', model);
  fd.append('conf', conf);

  try {
    const res  = await fetch('/predict/video', { method: 'POST', body: fd });
    const data = await res.json();
    if (data.error) { alert(data.error); return; }

    updateModelBadge(data.model_name);
    vidMeta.innerHTML = `ໂມເດລ: <strong>${data.model_name}</strong> &nbsp;·&nbsp; ${data.model_params}`;

    const totalSec = (data.total_ms / 1000).toFixed(1);
    const detTotal = Object.values(data.detections_summary).reduce((a,b) => a+b, 0);

    vidStats.innerHTML = `
      <div class="vstat-card"><div class="vstat-num">${data.frame_count.toLocaleString()}</div><div class="vstat-lbl">Frames</div></div>
      <div class="vstat-card"><div class="vstat-num">${totalSec}s</div><div class="vstat-lbl">ເວລາລວມ</div></div>
      <div class="vstat-card"><div class="vstat-num">${data.avg_ms_per_frame}</div><div class="vstat-lbl">ms/Frame</div></div>
      <div class="vstat-card"><div class="vstat-num">${detTotal}</div><div class="vstat-lbl">Detections</div></div>
    ` + Object.entries(data.detections_summary).map(([n,c]) =>
      `<div class="vstat-card"><div class="vstat-num" style="color:${detColor(n)}">${c}</div><div class="vstat-lbl">${n.toUpperCase()}</div></div>`
    ).join('');

    if (data.sample_frame) {
      vidSample.src = 'data:image/jpeg;base64,' + data.sample_frame;
      vidSampleWrap.classList.remove('hidden');
    } else {
      vidSampleWrap.classList.add('hidden');
    }
    vidResult.classList.remove('hidden');
  } catch(e) { alert('ຜິດພາດ: ' + e.message); }
  finally {
    vidBtn.disabled = false;
    vidBtn.textContent = 'ປະມວນຜົນວິດີໂອ';
    vidProgress.classList.add('hidden');
  }
});

// ===== WEBCAM =====
const camVideo   = document.getElementById('camVideo');
const camCanvas  = document.getElementById('camCanvas');
const camStart   = document.getElementById('camStart');
const camStop    = document.getElementById('camStop');
const camFps     = document.getElementById('camFps');
const camModelInfo = document.getElementById('camModelInfo');
const camError   = document.getElementById('camError');
const camHint    = document.getElementById('camHint');

let webcamRunning = false;
let webcamStream  = null;
let fpsSamples    = [];
const SEND_W      = 640;
const MIN_MS      = 300;

async function startCam() {
  camError.classList.add('hidden');
  camHint.classList.add('hidden');
  try {
    webcamStream = await navigator.mediaDevices.getUserMedia({
      video: { width: { ideal: 1280 }, height: { ideal: 720 } }, audio: false
    });
    camVideo.srcObject = webcamStream;
    await camVideo.play();
    webcamRunning = true;
    camStart.disabled = true;
    camStop.disabled  = false;
    fpsSamples = [];
    camLoop();
  } catch(err) {
    let msg = '❌ ບໍ່ສາມາດເປີດກ້ອງໄດ້: ' + err.message;
    if (err.name === 'NotAllowedError')  msg = '❌ ບໍ່ໄດ້ຮັບອະນຸຍາດ — ກະລຸນາອະນຸຍາດ Camera ໃນ Browser';
    if (err.name === 'NotFoundError')    msg = '❌ ບໍ່ພົບ Camera — ກະລຸນາຕໍ່ Webcam';
    if (err.name === 'NotReadableError') msg = '❌ Camera ກຳລັງຖືກໃຊ້ — ກະລຸນາປິດ App ອື່ນ';
    camError.textContent = msg;
    camError.classList.remove('hidden');
  }
}

function stopCam() {
  webcamRunning = false;
  if (webcamStream) { webcamStream.getTracks().forEach(t => t.stop()); webcamStream = null; }
  camVideo.srcObject = null;
  camCanvas.getContext('2d').clearRect(0, 0, camCanvas.width, camCanvas.height);
  camFps.textContent = '-- FPS';
  camStart.disabled = false;
  camStop.disabled  = true;
}

async function camLoop() {
  if (!webcamRunning) return;
  const t0 = performance.now();
  try {
    const ratio = camVideo.videoHeight / camVideo.videoWidth;
    const sendH = Math.round(SEND_W * ratio) || 480;
    const off = document.createElement('canvas');
    off.width = SEND_W; off.height = sendH;
    off.getContext('2d').drawImage(camVideo, 0, 0, SEND_W, sendH);
    const b64 = off.toDataURL('image/jpeg', 0.75);

    const model = document.getElementById('camModel').value;
    const conf  = document.getElementById('camConf').value;

    const resp = await fetch('/predict/webcam_frame', {
      method: 'POST',
      headers: { 'Content-Type': 'application/json' },
      body: JSON.stringify({ image: b64, model_name: model, conf: parseFloat(conf) }),
    });
    const data = await resp.json();

    const elapsed = performance.now() - t0;
    fpsSamples.push(elapsed);
    if (fpsSamples.length > 8) fpsSamples.shift();
    const avgMs = fpsSamples.reduce((a,b) => a+b, 0) / fpsSamples.length;
    const fps   = Math.round(1000 / avgMs);
    camFps.textContent      = `${fps} FPS · ${Math.round(data.infer_ms)}ms`;
    camModelInfo.textContent = `${data.model_name} · ${data.model_params}`;
    updateModelBadge(data.model_name);
    if (fps < 3) camHint.classList.remove('hidden');
    drawDets(data.detections, SEND_W, sendH);
  } catch(e) { console.warn('cam frame error:', e); }

  const spent = performance.now() - t0;
  const wait  = Math.max(0, MIN_MS - spent);
  if (webcamRunning) setTimeout(camLoop, wait);
}

function drawDets(dets, srcW, srcH) {
  const ctx = camCanvas.getContext('2d');
  camCanvas.width  = camVideo.clientWidth  || 640;
  camCanvas.height = camVideo.clientHeight || 480;
  ctx.clearRect(0, 0, camCanvas.width, camCanvas.height);
  if (!dets || !dets.length) return;
  const sx = camCanvas.width  / srcW;
  const sy = camCanvas.height / srcH;
  ctx.font = 'bold 13px Inter, monospace';
  for (const d of dets) {
    const [x1,y1,x2,y2] = d.box;
    const color = detColor(d.name);
    const label = `${d.name.toUpperCase()} ${Math.round(d.conf*100)}%`;
    ctx.strokeStyle = color; ctx.lineWidth = 2;
    ctx.strokeRect(x1*sx, y1*sy, (x2-x1)*sx, (y2-y1)*sy);
    const tw = ctx.measureText(label).width;
    ctx.fillStyle = color;
    ctx.fillRect(x1*sx, y1*sy - 20, tw + 10, 20);
    ctx.fillStyle = '#ffffff';
    ctx.fillText(label, x1*sx + 5, y1*sy - 5);
  }
}

camStart.addEventListener('click', startCam);
camStop.addEventListener('click',  stopCam);


## 6️⃣  ເລີ່ມ FastAPI Server + Ngrok Tunnel

> ⚠️ ຕ້ອງໄດ້ຕັ້ງ **Ngrok Auth Token** ກ່ອນ:
> 1. ສ້າງ Account ຟຣີທີ່ https://ngrok.com
> 2. ຄັດລອກ token ຈາກ Dashboard
> 3. ໃສ່ໃນ `NGROK_TOKEN = '...'` ດ້ານລຸ່ມ


In [ ]:
import threading, time, uvicorn
from pyngrok import ngrok, conf as ngrok_conf

# ===== ตั้งค่า Ngrok Token =====
NGROK_TOKEN = 'YOUR_NGROK_TOKEN_HERE'  # ← ใส่ token ของคุณที่นี่

if NGROK_TOKEN == 'YOUR_NGROK_TOKEN_HERE':
    print('⚠️  ກະລຸນາໃສ່ NGROK_TOKEN ກ່ອນ!')
else:
    ngrok_conf.get_default().auth_token = NGROK_TOKEN

    def run_server():
        uvicorn.run('main:app', host='0.0.0.0', port=8000, log_level='warning')

    t = threading.Thread(target=run_server, daemon=True)
    t.start()
    time.sleep(4)  # ລໍໃຫ້ server ເລີ່ມ

    tunnel = ngrok.connect(8000)
    url    = tunnel.public_url

    print('=' * 60)
    print('🚨  ລະບົບກວດຈັບຍານພາຫະນະສຸກເສີນ — ພ້ອມໃຊ້ງານ!')
    print('=' * 60)
    print(f'\n🌐  URL: {url}')
    print(f'\n   ✅  ເປີດ URL ດ້ານເທິງໃນ Browser')
    print(f'   ✅  ໜ້າ Home → ກວດຈັບ → Webcam → Stats')
    print('=' * 60)
